# Data Analysis Agent

> Run this notebook to launch the Web UI, then upload/drop a CSV in the browser.

## Quick Start
1. Run Cell 2 (setup helpers).
2. Run Cell 3 (start server + clickable URL).
3. Open the URL, upload CSV, type question, and click Analyze Dataset.
4. (Optional) Run Cell 4 to stop the server.

In [7]:
import os
import sys
import subprocess
import time
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

from IPython.display import HTML, display

HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}"

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app.py").exists() and (candidate / "static").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing app.py and static/")

def _wait_for_server(url: str, timeout_seconds: int = 30) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with urlopen(f"{url}/health", timeout=2) as response:
                if response.status == 200:
                    return True
        except URLError:
            pass
        time.sleep(1)
    return False

PROJECT_ROOT = _find_project_root(Path.cwd())
SERVER_PROCESS = None

print(f"Project root: {PROJECT_ROOT}")
print(f"Web UI will run at: {BASE_URL}")

Project root: c:\Users\hj152\OneDrive\文档\GitHub\CS539_Project\CS539_Project
Web UI will run at: http://127.0.0.1:8000


In [8]:
# Start Web UI server
if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
    print(f"Server is already running at {BASE_URL}")
else:
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        api_key = getpass("Enter Gemini API key for Web UI: ").strip()
        if not api_key:
            raise ValueError("Gemini API key is required to start analysis server.")

    server_env = os.environ.copy()
    server_env["GEMINI_API_KEY"] = api_key

    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", HOST, "--port", str(PORT)],
        cwd=str(PROJECT_ROOT),
        env=server_env,
    )

    if not _wait_for_server(BASE_URL, timeout_seconds=30):
        raise RuntimeError("Server did not become ready in time. Check notebook output for errors.")

display(HTML(f'<a href="{BASE_URL}" target="_blank">Open Data Analysis Web UI</a>'))
print(f"Web UI ready: {BASE_URL}")
print("Now upload/drop a CSV in the browser and click Analyze Dataset.")

Web UI ready: http://127.0.0.1:8000
Now upload/drop a CSV in the browser and click Analyze Dataset.


In [3]:
# Optional: stop Web UI server
if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=10)
    print("Web UI server stopped.")
else:
    print("No running Web UI server found.")

Web UI server stopped.
